# Lesson 2: Static Scraping with requests & BeautifulSoup

**Week 5 · Data Engineering Course**

---

In Lesson 1 you parsed HTML that was already on your disk. Now you will go and get it.

A **static** site sends you the finished HTML: everything you see in the browser is in the response body. `requests` is enough for those, and they are still the majority of the pages a data engineer scrapes.

This lesson teaches you:
- How to fetch HTML with `requests` (and the encoding bug that bites everyone once)
- How to build a polite, reusable `fetch()` function with headers, timeouts, and retries
- How to scrape a full page of records into a DataFrame
- How to follow **pagination** until the pages run out
- How to follow links to **detail pages** and join the two levels together
- How to cache raw HTML so you stop re-hitting the server
- How to lift HTML `<table>` elements straight into pandas with `pd.read_html()`

**Install before starting:**
```
pip install requests beautifulsoup4 lxml pandas
```

### Our practice site

We scrape **[books.toscrape.com](https://books.toscrape.com)** — a bookshop built by Zyte specifically so people can learn to scrape it. 1,000 books across 50 pages, each with a detail page. Nobody minds how often you hit it, and the HTML has never changed.

Scraping a purpose-built sandbox is the right way to learn. Point your first scraper at a real company's site and you will be debugging a ban at the same time as a selector.

In [ ]:
!pip install requests beautifulsoup4 lxml pandas

In [ ]:
import re
import time
import requests
import pandas as pd
from io import StringIO
from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DATA  = Path('data')
PAGES = DATA / 'pages'
RAW   = DATA / 'raw_html'       # a cache for pages we have already fetched
CLEAN = DATA / 'clean'
for p in [RAW, CLEAN]:
    p.mkdir(parents=True, exist_ok=True)

BASE = 'https://books.toscrape.com/'
print('Ready.')

---

## 2.1 Fetching HTML

This is the same `requests.get()` you used for APIs last week. The only difference is what you do with the response:

| Last week (API) | This week (HTML) |
|-----------------|------------------|
| `response.json()` | `response.text` |
| returns a dict / list | returns a string of HTML |
| parse with `[]` keys | parse with BeautifulSoup |

`response.text` is the decoded string. `response.content` is the raw bytes. The difference matters, as the next cell shows.

In [ ]:
response = requests.get(BASE, timeout=10)

print(f'status code:      {response.status_code}')
print(f'content-type:     {response.headers.get("content-type")}')
print(f'characters:       {len(response.text):,}')
print(f'bytes:            {len(response.content):,}')
print()
print(response.text[:300])

### The `Â£` bug — read this before you lose an hour to it

Look at a price on that page and you will see `Â£51.77` instead of `£51.77`.

Nothing is wrong with the site. The server sent UTF-8 bytes but did not declare a charset in its `Content-Type` header, and the HTTP standard says "no charset declared" means ISO-8859-1. `requests` follows the standard, decodes UTF-8 bytes as ISO-8859-1, and you get mojibake: one character becomes two.

Two fixes:

```python
response.encoding = 'utf-8'                     # you know what it is — say so
response.encoding = response.apparent_encoding  # let chardet sniff it
```

Set it **before** you touch `response.text`. The decoding happens on access.

> **Rule:** when scraped text contains `Â`, `â€™`, or `Ã©`, it is an encoding problem, not a parsing problem. Never try to fix it with `str.replace()` — fix the decoding.

In [ ]:
# The bug
response = requests.get(BASE, timeout=10)
print('requests guessed:  ', response.encoding)
print('chardet thinks:    ', response.apparent_encoding)

soup_bad = BeautifulSoup(response.text, 'html.parser')
print('price (broken):    ', soup_bad.select_one('.price_color').get_text(strip=True))

# The fix — set the encoding, then read .text
response.encoding = 'utf-8'
soup_ok = BeautifulSoup(response.text, 'html.parser')
print('price (fixed):     ', soup_ok.select_one('.price_color').get_text(strip=True))

---

## 2.2 A Polite `fetch()` Function

Every request your scraper makes should carry the same four things. Write them once, in a function, and never think about them again:

1. **A `User-Agent` header.** The default `python-requests/2.x` is an invitation to be blocked. Say who you are and how to reach you.
2. **A timeout.** Without one, a hung server hangs your pipeline forever. There is no default.
3. **`raise_for_status()`.** A `404` still returns a response object with a body — parsing it gives you confusing selector failures instead of a clear error.
4. **A `Session`.** It reuses the underlying TCP connection across requests, which is both faster for you and lighter on the server.

Add retries with backoff for transient failures (`429`, `500`, `503`) and you have a fetcher that survives a real run.

In [ ]:
HEADERS = {
    # Identify your scraper honestly. Put a real contact address in a real project.
    'User-Agent': 'DE-Course-Scraper/1.0 (student project; contact: you@example.com)',
    'Accept-Language': 'en-GB,en;q=0.9',
}

session = requests.Session()
session.headers.update(HEADERS)

RETRY_ON = {429, 500, 502, 503, 504}


def fetch(url, *, encoding='utf-8', retries=3, pause=1.0):
    '''Fetch a URL and return a BeautifulSoup tree.

    Retries transient server errors with a growing backoff, sets the
    encoding explicitly, and raises on any other HTTP error.
    '''
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=15)
        except requests.RequestException as e:          # DNS, connection, timeout
            if attempt == retries:
                raise
            print(f'  [{attempt}/{retries}] {type(e).__name__} — retrying')
            time.sleep(pause * attempt)
            continue

        if r.status_code in RETRY_ON and attempt < retries:
            print(f'  [{attempt}/{retries}] HTTP {r.status_code} — backing off')
            time.sleep(pause * 2 ** attempt)            # 2s, 4s, 8s
            continue

        r.raise_for_status()                            # 404, 403, ... -> exception
        r.encoding = encoding
        return BeautifulSoup(r.text, 'html.parser')

    raise RuntimeError(f'Gave up on {url} after {retries} attempts')


soup = fetch(BASE)
print(soup.title.get_text(strip=True))

In [ ]:
# raise_for_status() turns a bad URL into an obvious failure instead of a silent one
try:
    fetch(BASE + 'no-such-page.html')
except requests.HTTPError as e:
    print(f'HTTPError: {e}')

print()
print('Without raise_for_status you would get a 404 page, parse it happily,')
print('and spend an hour wondering why your selector returns an empty list.')

---

## 2.3 Scraping One Page of Records

Same method as Lesson 1: find the repeating element, then pull the same fields from each one.

Inspect a book on books.toscrape.com and you find:

```html
<article class="product_pod">
  <div class="image_container">
    <a href="catalogue/a-light-in-the-attic_1000/index.html"><img ... alt="A Light in the Attic"></a>
  </div>
  <p class="star-rating Three"> ... </p>                        <!-- rating is in the CLASS -->
  <h3><a href="catalogue/..." title="A Light in the Attic">A Light in the ...</a></h3>
  <div class="product_price">
    <p class="price_color">£51.77</p>
    <p class="instock availability"> In stock </p>
  </div>
</article>
```

Three things worth noticing, because all three are common in the wild:

- **The visible title is truncated** (`A Light in the ...`) but the full title is in the `title` **attribute**. Prefer the attribute.
- **The rating is encoded as a class name** — `class="star-rating Three"`. There is no text to read; you take `['class'][1]` and map the word to a number.
- **The price is a string with a currency symbol.** It is text until you clean it.

In [ ]:
products = soup.select('article.product_pod')
print(f'{len(products)} books on this page\n')
print(products[0].prettify())

In [ ]:
# Work out each field on ONE record first
p = products[0]

print('title (text):    ', p.select_one('h3 a').get_text(strip=True))      # truncated
print('title (attr):    ', p.select_one('h3 a')['title'])                  # full
print('href:            ', p.select_one('h3 a')['href'])                   # relative!
print('price:           ', p.select_one('.price_color').get_text(strip=True))
print('availability:    ', p.select_one('.availability').get_text(strip=True))
print('rating classes:  ', p.select_one('p.star-rating')['class'])
print('image alt:       ', p.select_one('img')['alt'])

### Relative URLs and `urljoin`

`href="catalogue/a-light-in-the-attic_1000/index.html"` is **relative** — it only means something next to the URL you found it on. And the same link is written differently depending on which page you are standing on:

| Found on | `href` in the HTML |
|----------|--------------------|
| `books.toscrape.com/` | `catalogue/a-light-in-the-attic_1000/index.html` |
| `books.toscrape.com/catalogue/page-2.html` | `in-her-wake_980/index.html` |

Never build URLs with string concatenation or f-strings. `urljoin(page_url, href)` applies the real browser rules — relative paths, `../`, leading `/`, and already-absolute URLs all come out correct.

```python
urljoin('https://books.toscrape.com/catalogue/page-2.html', 'in-her-wake_980/index.html')
# -> 'https://books.toscrape.com/catalogue/in-her-wake_980/index.html'
```

In [ ]:
# urljoin resolves every form of href correctly
page_url = 'https://books.toscrape.com/catalogue/page-2.html'

for href in ['in-her-wake_980/index.html',
             '../index.html',
             '/catalogue/page-3.html',
             'https://example.com/elsewhere']:
    print(f'{href:<42} -> {urljoin(page_url, href)}')

In [ ]:
RATINGS = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


def text_at(node, selector, default=None):
    '''Stripped text of the first match, or `default` if there is no match.'''
    found = node.select_one(selector)
    return found.get_text(strip=True) if found else default


def parse_price(text):
    '''"£51.77" -> 51.77 ; None -> None. Works for "$1,299.00" too.'''
    if not text:
        return None
    digits = re.sub(r'[^\d.]', '', text.replace(',', ''))
    return float(digits) if digits else None


def parse_stock(text):
    '''"In stock (22 available)" -> 22 ; "In stock" -> None.'''
    m = re.search(r'(\d+)', text or '')
    return int(m.group(1)) if m else None


def parse_product(pod, page_url):
    '''Turn one article.product_pod into a dict.'''
    link   = pod.select_one('h3 a')
    rating = pod.select_one('p.star-rating')
    return {
        'title':     link['title'] if link else None,
        'price_gbp': parse_price(text_at(pod, '.price_color')),
        'rating':    RATINGS.get(rating['class'][1]) if rating else None,
        'in_stock':  'in stock' in (text_at(pod, '.availability', '')).lower(),
        'url':       urljoin(page_url, link['href']) if link else None,
    }


parse_product(products[0], BASE)

In [ ]:
rows = [parse_product(pod, BASE) for pod in products]
books = pd.DataFrame(rows)

print(books.dtypes)
books.head(10)

---

## 2.4 Pagination

One page of 20 books is a demo. A dataset is all 50 pages.

There are two ways to walk a paginated site, and you should know both:

**A. Guess the URL pattern.** `catalogue/page-1.html`, `page-2.html`, ... Fast, obvious, and you can parallelise it — but you need to know where to stop, and the pattern changes without warning.

**B. Follow the "next" link.** Read `li.next a` off each page and `urljoin` it. This is how a human navigates, it stops naturally when the link disappears, and it survives a change of URL scheme. Prefer this.

Either way, three rules:

- **Sleep between requests.** `time.sleep(1)` is the cheapest insurance you will ever buy.
- **Cap the loop.** A `while True` with a bug is a denial-of-service attack you wrote by accident. Always pass `max_pages`.
- **Log progress.** A 50-page scrape that prints nothing is indistinguishable from a hung one.

In [ ]:
def scrape_page(url):
    '''Scrape one listing page. Returns (rows, next_page_url_or_None).'''
    soup = fetch(url)
    rows = [parse_product(pod, url) for pod in soup.select('article.product_pod')]

    next_link = soup.select_one('li.next a')
    next_url  = urljoin(url, next_link['href']) if next_link else None
    return rows, next_url


rows, next_url = scrape_page(BASE)
print(f'{len(rows)} rows from page 1')
print(f'next page: {next_url}')

In [ ]:
def scrape_all(start_url, max_pages=3, pause=1.0):
    '''Follow the "next" link until it runs out or we hit max_pages.'''
    all_rows, url, page = [], start_url, 1

    while url and page <= max_pages:
        print(f'page {page}: {url}')
        rows, url = scrape_page(url)
        all_rows.extend(rows)
        print(f'         +{len(rows)} rows (total {len(all_rows)})')

        if url:
            time.sleep(pause)        # be polite: one request per second
        page += 1

    if url:
        print(f'stopped at max_pages={max_pages}; more pages remain')
    return pd.DataFrame(all_rows)


# 3 pages = 60 books. Raise max_pages to 50 to take the whole site.
books = scrape_all(BASE, max_pages=3)
print(f'\nScraped {len(books)} books')
books.head()

In [ ]:
# Sanity-check the harvest before you trust it
print('rows:              ', len(books))
print('unique titles:     ', books["title"].nunique())
print('missing values:')
print(books.isna().sum())
print()
print(books[['price_gbp', 'rating']].describe())

---

## 2.5 Detail Pages

Listing pages give you a shallow row per record. The interesting fields — description, category, stock count, UPC — live on each record's own page.

The pattern is a **two-level scrape**:

```
1. scrape the listing   -> N rows, each with a url
2. for each url: fetch  -> enrich that row with detail fields
```

The cost is one request per record. 1,000 books at one second each is 17 minutes — acceptable. 100,000 records is not, so in production you would limit the detail pass to rows you actually need, and cache every page you fetch.

> **Always cap your detail pass while developing.** Debug against 5 records, not 1,000. The code below uses `.head(5)`.

In [ ]:
def scrape_book_detail(url):
    '''Fetch one book's page and pull the fields the listing does not carry.'''
    soup = fetch(url)

    # The description is a <p> that SITS AFTER the #product_description heading,
    # not inside it — a classic case for find_next_sibling().
    heading = soup.select_one('#product_description')
    description = heading.find_next_sibling('p').get_text(strip=True) if heading else None

    # breadcrumb: Home > Books > Poetry  -> the category is the second-to-last item
    crumbs  = [a.get_text(strip=True) for a in soup.select('ul.breadcrumb li a')]
    category = crumbs[-1] if len(crumbs) > 1 else None

    # The product information table is a <th>/<td> key-value table
    info = {
        tr.th.get_text(strip=True): tr.td.get_text(strip=True)
        for tr in soup.select('table.table-striped tr')
        if tr.th and tr.td
    }

    return {
        'url':         url,
        'category':    category,
        'upc':         info.get('UPC'),
        'stock_count': parse_stock(info.get('Availability')),
        'n_reviews':   int(info.get('Number of reviews', 0) or 0),
        'description': description,
    }


detail = scrape_book_detail(books.loc[0, 'url'])
for k, v in detail.items():
    print(f'{k:<13}: {str(v)[:95]}')

In [ ]:
# Enrich the first 5 rows, then join the two levels on `url`
sample  = books.head(5)
details = []

for url in sample['url']:
    details.append(scrape_book_detail(url))
    print(f'fetched {url.split("/")[-2]}')
    time.sleep(1)

detail_df = pd.DataFrame(details)
enriched  = sample.merge(detail_df, on='url', how='left')

enriched[['title', 'price_gbp', 'rating', 'category', 'stock_count', 'n_reviews']]

---

## 2.6 Cache the Raw HTML

While you are getting a selector right you will run the same cell twenty times. Twenty runs should be **one** request, not twenty.

Save every page you fetch to disk, keyed by URL, and read from the cache on the next run. This is not a nicety — it is the single biggest difference between a scraper that is pleasant to develop and one that gets you blocked:

- iteration becomes instant and works offline
- the site sees one request instead of twenty
- when a selector fails you still have the exact HTML that broke it, which you cannot get back from a live site

Keep the raw HTML out of Git (it is large and reproducible) — add `data/raw_html/` to your `.gitignore`.

In [ ]:
import hashlib


def cache_path(url):
    '''A stable, filesystem-safe filename for a URL.'''
    digest = hashlib.md5(url.encode()).hexdigest()[:10]
    slug   = re.sub(r'[^a-z0-9]+', '-', url.lower().split('//')[-1])[:60].strip('-')
    return RAW / f'{slug}-{digest}.html'


def cached_get(url, encoding='utf-8', pause=1.0):
    '''Return the HTML for `url`, fetching it only if it is not already on disk.'''
    path = cache_path(url)
    if path.exists():
        print(f'cache hit:  {path.name}')
        return path.read_text(encoding='utf-8')

    print(f'fetching:   {url}')
    r = session.get(url, timeout=15)
    r.raise_for_status()
    r.encoding = encoding
    path.write_text(r.text, encoding='utf-8')
    time.sleep(pause)
    return r.text


html = cached_get(BASE)                       # first call: fetches and saves
html = cached_get(BASE)                       # second call: no request at all
print(f'\n{len(html):,} characters')
print('cached files:', [p.name for p in RAW.glob("*.html")])

---

## 2.7 Shortcut: `pd.read_html()` for Tables

If the data you want is already an HTML `<table>`, you do not need BeautifulSoup at all. `pd.read_html()` finds every table on a page and returns a **list of DataFrames**.

```python
pd.read_html(path_or_url)            # a file path or a URL
pd.read_html(StringIO(html_string))  # an HTML string MUST be wrapped in StringIO
```

> **pandas 2+ requires `StringIO`** around raw HTML strings. Passing the string directly makes pandas treat it as a path and raise a confusing `FileNotFoundError: [Errno 2] No such file or directory: '<!DOCTYPE html>...'`. If you see that error, you forgot the wrapper.

Use `match='some text'` to pick the table you want instead of counting indexes — index 3 becomes index 4 the day the page gains a table.

This only works for genuine `<table>` markup. A grid of `<div>`s that *looks* like a table is invisible to `read_html`.

In [ ]:
# From a local file: pass the path directly
tables = pd.read_html(PAGES / 'salaries_table.html')
print(f'{len(tables)} tables found in salaries_table.html\n')

for i, t in enumerate(tables):
    print(f'--- table {i}: {t.shape} ---')
    print(t)
    print()

In [ ]:
# Pick a table by its content, not its position
roles = pd.read_html(PAGES / 'salaries_table.html', match='Open roles')[0]
print(roles)

# From a live page: the book detail page's product information table
# Note the StringIO wrapper — required for an HTML string.
r = session.get(books.loc[0, 'url'], timeout=15)
r.encoding = 'utf-8'

info_table = pd.read_html(StringIO(r.text))[0]
print()
print(info_table)

---

## 2.8 When Static Scraping Fails

Everything in this lesson relies on one assumption: **the data is in the HTML the server sent.**

Increasingly it is not. Modern sites ship a nearly empty HTML shell plus JavaScript that fetches the data and builds the page in your browser. `requests` does not run JavaScript, so it receives the shell — your selector matches nothing, even though you can see the data on screen with your own eyes.

`quotes.toscrape.com` publishes the same content twice so you can see the difference. Run the next cell.

In [ ]:
# Same content, two pages: one rendered by the server, one by JavaScript
for url in ['https://quotes.toscrape.com/', 'https://quotes.toscrape.com/js/']:
    s = fetch(url)
    quotes = s.select('.quote')
    print(f'{url:<36} .quote elements found: {len(quotes)}')

print()
print('The /js/ page looks identical in a browser but is empty to requests.')
print('Its quotes are written by a <script> tag that only a browser executes:')
print()
js_soup = fetch('https://quotes.toscrape.com/js/')
print(js_soup.select_one('script:-soup-contains("document.write")').get_text()[:260], '...')

**That is the wall.** No selector, parser, or header fixes it — the data was never in the response.

The fix is to use a real browser engine that runs the JavaScript, then read the page it built. That is **Playwright**, and it is Lesson 3.

### How to tell before you waste an hour

| Check | Static | JavaScript-rendered |
|-------|--------|---------------------|
| **View Page Source** (Ctrl+U) — search for a value you can see | found | not found |
| `len(soup.select('your-selector'))` | > 0 | 0 |
| DevTools → Network → **Fetch/XHR** while reloading | nothing interesting | JSON responses carrying your data |

That last row is worth a habit: when a page loads its data over XHR, the browser is calling an **API**. Open the Network tab, find the JSON request, and call that URL with `requests` instead. You get clean JSON, no browser, no parsing — the fastest scraper is often the one that skips scraping entirely.

In [ ]:
# Save the harvest — Lesson 4 compares the formats you could save it in
out = CLEAN / 'books.csv'
books.to_csv(out, index=False)
print(f'Saved {len(books)} rows to {out}')

enriched.to_csv(CLEAN / 'books_enriched.csv', index=False)
print(f'Saved {len(enriched)} enriched rows to {CLEAN / "books_enriched.csv"}')

pd.read_csv(out).head(3)

---

## Key Takeaways

1. HTML scraping is `response.text` where an API gave you `response.json()`. Everything else you learned about `requests` last week still applies.
2. **Set the encoding explicitly** (`response.encoding = 'utf-8'`) before reading `.text`. Text full of `Â` or `â€™` is a decoding bug, never something to patch with `str.replace()`.
3. Every request needs a **`User-Agent`**, a **`timeout`**, and **`raise_for_status()`**. Put them in one `fetch()` function and reuse it; a `Session` makes the whole run faster and lighter on the server.
4. **Retry transient errors** (`429`, `5xx`) with a growing backoff. Let real errors (`404`, `403`) raise — a silently parsed error page is the worst outcome.
5. Scrape a listing by finding the **repeating element**, then extracting the same fields from each. Some fields hide in attributes (`['title']`, `['href']`) or even in **class names** (`class="star-rating Three"`).
6. **Always `urljoin(page_url, href)`.** The same link is written differently on different pages; string concatenation gets it wrong eventually.
7. **Follow the "next" link** rather than guessing URL patterns, and always pass a `max_pages` cap. `time.sleep(1)` between requests, every time.
8. A **two-level scrape** (listing → detail page) costs one request per record. Cap it while developing — debug against 5 rows, not 1,000.
9. **Cache raw HTML to disk.** Iterating on a saved file is instant, works offline, sends no traffic, and preserves the exact page that broke your selector. Keep the cache out of Git.
10. If the data is already in a `<table>`, `pd.read_html()` is one line. Wrap HTML **strings** in `StringIO`, and select with `match='text'` rather than a positional index.
11. If `requests` returns a page where your selector finds nothing that you can plainly see in the browser, the page is **JavaScript-rendered**. Check View Page Source to confirm.
12. Before reaching for a browser, open **DevTools → Network → Fetch/XHR**. If the page fetches its own data as JSON, call that endpoint directly — no scraping required.

---

**Next:** Lesson 3 — driving a real browser with Playwright to scrape pages that build themselves.